In [ ]:
# run this notebook after 9_find_differences_trios_python
# use this notebook to call DNMs in siblings
# after this, run 11_trim_ibd2_python

In [ ]:
# import hail 
import hail as hl
import os
import pandas
from google.cloud import storage
import re
import numpy as np 

In [ ]:
bucket=os.getenv('WORKSPACE_BUCKET')
hl.init(idempotent=True)
hl.default_reference('GRCh38')

In [ ]:
# put snipar ibd2 calls into bucket for hail to see beforehand
f_ibd2 = bucket + "/data/ibd_all_dec20.txt"
ibd2_all = pandas.read_table(f_ibd2)
ibd2_all["ID1"] = ibd2_all["ID1"].astype(str)
ibd2_all["ID2"] = ibd2_all["ID2"].astype(str)

In [ ]:
# now, go through each row in the file
for k in range(1,22):
    # read twins 
    f_sibs = f'{bucket}/data/sibs_samples_{k}.txt'
    sibs_all = pandas.read_table(f_sibs, sep = ',')
    sibs_all['idv1'] = sibs_all['idv1'].astype(str)
    sibs_all['idv2'] = sibs_all['idv2'].astype(str)
    merged = sibs_all.merge(finished, on=['idv1', 'idv2'], how='left', indicator=True)
    # Keep only rows that do not appear in 'finished'
    sibs_filtered1 = merged[merged['_merge'] == 'left_only'].drop(columns=['_merge'])
    
    sibs_filtered = sibs_filtered1.merge(quad, on=['idv1', 'idv2'], how='inner')

    print(sibs_filtered.shape)
    # read mt sub 
    mt_path =f'{bucket}/data/sibs_samples_{k}.mt'
    mt = hl.read_matrix_table(mt_path)
    # now parse 
    for j in range(sibs_filtered.shape[0]):
        # sib pairs
        idv1 = sibs_filtered["idv1"].iloc[j]
        idv2 = sibs_filtered["idv2"].iloc[j]
        sibs = [str(idv1), str(idv2)]
        # subset to samples 
        mt_subset_sib = mt.filter_cols(hl.literal(sibs).contains(mt.s))   
        
        # subset to ibd2 regions 
        # ibd2 pertaining to this pair
        ibd2_sub = ibd2_all[(ibd2_all["ID1"] == idv1) & (ibd2_all["ID2"] == idv2)]
        chrs = ibd2_sub["Chr"].astype(int).astype(str).tolist()
        starts = ibd2_sub["start_coordinate"].astype(int).astype(str).tolist()
        ends = ibd2_sub["stop_coordinate"].astype(int).astype(str).tolist()
        intervals = [f"chr{c}:{start}-{end}" for c, start, end in zip(chrs, starts, ends)]
        
        if len(intervals) > 0:      
            mt_subset_sib = hl.filter_intervals(mt_subset_sib,[hl.parse_locus_interval(x,) for x in intervals])

            # remove loci with missing genotypes
            mt_subset_sib = mt_subset_sib.annotate_entries(GT = hl.or_missing(mt_subset_sib.GQ >= 30, mt_subset_sib.GT))
            mt_subset_sib = mt_subset_sib.filter_rows(hl.agg.sum(hl.is_defined(mt_subset_sib.GT)) == 2, keep = True)

            # keep anything nonref 
            mt_subset_sib = mt_subset_sib.filter_rows(hl.agg.any(mt_subset_sib.GT.is_non_ref()))

            # subset to those that have a passing FT 
            mt_subset_sib = mt_subset_sib.filter_rows(hl.agg.sum(mt_subset_sib.FT == "FAIL") == 0, keep = True)

            # repartition to join faster 
            mt_subset_sib = mt_subset_sib.repartition(50)
        
            mt_subset_sib.GT.export(f'{bucket}/sibs/difs_all_alt/{idv1}_{idv2}_difs_all_alt.tsv')
            

In [ ]:
%%bash 

mkdir -p sibs 

gsutil -m cp -r $WORKSPACE_BUCKET/sibs/difs_all_alt ./sibs/

mkdir -p ./sibs/formatted_difs
mkdir -p ./sibs/quads_formatted_difs